## 1. 관찰 종료일 · 시간대 확정

기존 프로젝트와 동일하게 `2023-07-31`까지 사용합니다. **시간대는 KST(Asia/Seoul)로 통일**합니다
(프로젝트 전반의 `_kst` 컬럼 관례와 일치시킴). 종료 시각은 "다음 날 0시 미만"으로 명시해서 경계를
명확히 합니다.

```
관찰 종료일: 2023-07-31 (KST)
시간대: KST (Asia/Seoul)
T0: 사용자별 최초 PING 발신 시각 (first_ping_at, KST 기준)
피처 구간: T0 이상 ~ T0+24시간 미만
타깃 구간: T0+7일 이상 ~ T0+8일 미만
모집단: 최초 PING 발신 기록이 있고, T0+8일까지 완전히 관찰 가능한 사용자
```

⚠️ 원천 테이블(`created_at`)이 UTC로 저장돼 있다는 전제 하에, 비교 경계값만 KST 기준 벽시계 시각으로
정의합니다 (`TIMESTAMP(..., 'Asia/Seoul')`). `TIMESTAMP`는 내부적으로 절대 시점이라 비교 자체는
시간대와 무관하지만, 사람이 읽는 `observation_end_exclusive` 값의 의미를 KST로 맞추는 것이 핵심입니다.


## 0. 환경설정

In [ ]:
from google.colab import auth
auth.authenticate_user()

%load_ext google.cloud.bigquery

## 1. 관찰 종료일 · 시간대 확정

기존 프로젝트와 동일하게 `2023-07-31`까지 사용합니다. 종료 시각은 "다음 날 0시 미만"으로 명시해서
경계를 명확히 합니다 (원천 데이터가 `+00:00` UTC로 출력되므로 아래는 UTC 기준).

```
관찰 종료일: 2023-07-31
시간대: UTC
T0: 사용자별 최초 PING 발신 시각 (first_ping_at)
피처 구간: T0 이상 ~ T0+24시간 미만
타깃 구간: T0+7일 이상 ~ T0+8일 미만
모집단: 최초 PING 발신 기록이 있고, T0+8일까지 완전히 관찰 가능한 사용자
```


## 2. 최종 후보 N명 계산
**Base 생성 쿼리보다 이 확인 쿼리를 먼저 돌립니다.** N명을 확인한 뒤 팀에 공유하고, 그 다음 Base SQL을 실행합니다.

In [ ]:
%%bigquery df_eligible_check --project your-gcp-project

WITH params AS (
  SELECT
    TIMESTAMP('2023-08-01 00:00:00', 'Asia/Seoul') AS observation_end_exclusive
),

first_ping AS (
  SELECT
    CAST(q.user_id AS INT64) AS user_id,
    MIN(q.created_at) AS first_ping_at
  FROM `your-gcp-project.raw.raw_userquestionrecord` q
  CROSS JOIN params p
  WHERE q.user_id IS NOT NULL
    -- 기존 프로젝트 관찰기간까지만 사용
    AND q.created_at < p.observation_end_exclusive
  GROUP BY CAST(q.user_id AS INT64)
)

SELECT
  COUNT(*) AS ping_sender_count,

  COUNTIF(
    TIMESTAMP_ADD(first_ping_at, INTERVAL 8 DAY) <= observation_end_exclusive
  ) AS eligible_user_count,

  COUNTIF(
    TIMESTAMP_ADD(first_ping_at, INTERVAL 8 DAY) > observation_end_exclusive
  ) AS excluded_incomplete_count

FROM first_ping
CROSS JOIN params;

In [ ]:
df_eligible_check
print(f"전체 PING 발신자: {df_eligible_check['ping_sender_count'][0]:,}명")
print(f"완전 관찰 가능: {df_eligible_check['eligible_user_count'][0]:,}명")
print(f"관찰 부족 제외: {df_eligible_check['excluded_incomplete_count'][0]:,}명")

**확인할 값**
- `ping_sender_count`: 관찰기간 안에 PING을 보낸 전체 사용자
- `eligible_user_count`: D+7 모델에 실제 사용할 N명
- `excluded_incomplete_count`: D+8까지 못 봐서 제외된 사용자

이 결과를 팀에 먼저 공유하세요: `전체 PING 발신자: __명 / 완전 관찰 가능: N명 / 관찰 부족 제외: __명`

## 3. N명이 이상하지 않은지 점검

In [ ]:
%%bigquery df_sanity_check --project your-gcp-project

WITH params AS (
  SELECT
    TIMESTAMP('2023-08-01 00:00:00', 'Asia/Seoul') AS observation_end_exclusive
),

first_ping AS (
  SELECT
    CAST(q.user_id AS INT64) AS user_id,
    MIN(q.created_at) AS first_ping_at
  FROM `your-gcp-project.raw.raw_userquestionrecord` q
  CROSS JOIN params p
  WHERE q.user_id IS NOT NULL
    AND q.created_at < p.observation_end_exclusive
  GROUP BY CAST(q.user_id AS INT64)
)

SELECT
  DATETIME(MIN(first_ping_at), 'Asia/Seoul') AS min_first_ping_at_kst,
  DATETIME(MAX(first_ping_at), 'Asia/Seoul') AS max_first_ping_at_kst,

  COUNTIF(
    TIMESTAMP_ADD(first_ping_at, INTERVAL 8 DAY) <= observation_end_exclusive
  ) AS eligible_user_count

FROM first_ping
CROSS JOIN params;

In [ ]:
df_sanity_check

**확인할 것**
- 최초 PING 시작일이 데이터 기간과 맞는지
- 마지막 최초 PING 시각이 대략 7월 24일 이전인지 (7/31 - 8일 = 7/23 근처가 정상)
- 사용자 수가 지나치게 적지 않은지

## 4. Base Mart 생성

In [ ]:
%%bigquery --project your-gcp-project

CREATE OR REPLACE TABLE `your-gcp-project.mart.mart_ml_d7_base` AS

WITH params AS (
  SELECT
    TIMESTAMP('2023-08-01 00:00:00', 'Asia/Seoul') AS observation_end_exclusive
),

first_ping AS (
  SELECT
    CAST(q.user_id AS INT64) AS user_id,
    MIN(q.created_at) AS first_ping_at
  FROM `your-gcp-project.raw.raw_userquestionrecord` q
  CROSS JOIN params p
  WHERE q.user_id IS NOT NULL
    AND q.created_at < p.observation_end_exclusive
  GROUP BY CAST(q.user_id AS INT64)
),

user_windows AS (
  SELECT
    f.user_id,
    f.first_ping_at,

    -- Feature 관찰 구간: T0 ~ T0+24h
    f.first_ping_at AS feature_start_at,
    TIMESTAMP_ADD(f.first_ping_at, INTERVAL 24 HOUR) AS feature_end_at,

    -- Target 관찰 구간: T0+7d ~ T0+8d
    TIMESTAMP_ADD(f.first_ping_at, INTERVAL 7 DAY) AS target_start_at,
    TIMESTAMP_ADD(f.first_ping_at, INTERVAL 8 DAY) AS target_end_at,

    p.observation_end_exclusive

  FROM first_ping f
  CROSS JOIN params p
),

/* -----------------------------------------
D+8까지 완전히 관찰 가능한 사용자만 포함
----------------------------------------- */
eligible_users AS (
  SELECT
    user_id,
    first_ping_at,
    feature_start_at,
    feature_end_at,
    target_start_at,
    target_end_at,
    observation_end_exclusive
  FROM user_windows
  WHERE target_end_at <= observation_end_exclusive
),

/* -----------------------------------------
D+7 구간 PING 활동 여부 생성
----------------------------------------- */
d7_target AS (
  SELECT
    e.user_id,
    CASE WHEN COUNT(q.user_id) > 0 THEN 1 ELSE 0 END AS retained_d7
  FROM eligible_users e
  LEFT JOIN `your-gcp-project.raw.raw_userquestionrecord` q
    ON e.user_id = CAST(q.user_id AS INT64)
    -- D+7 구간: 시작 포함, 종료 미포함
    AND q.created_at >= e.target_start_at
    AND q.created_at < e.target_end_at
  GROUP BY e.user_id
)

/* -----------------------------------------
최종 Base Mart
(TIMESTAMP 컬럼은 절대 시점 그대로 저장하고, 사람이 읽기 쉽도록 _kst 컬럼을 별도로 추가)
----------------------------------------- */
SELECT
  e.user_id,

  e.first_ping_at,
  DATETIME(e.first_ping_at, 'Asia/Seoul') AS first_ping_at_kst,

  e.feature_start_at,
  DATETIME(e.feature_start_at, 'Asia/Seoul') AS feature_start_at_kst,

  e.feature_end_at,
  DATETIME(e.feature_end_at, 'Asia/Seoul') AS feature_end_at_kst,

  e.target_start_at,
  DATETIME(e.target_start_at, 'Asia/Seoul') AS target_start_at_kst,

  e.target_end_at,
  DATETIME(e.target_end_at, 'Asia/Seoul') AS target_end_at_kst,

  e.observation_end_exclusive,
  DATETIME(e.observation_end_exclusive, 'Asia/Seoul') AS observation_end_exclusive_kst,

  COALESCE(t.retained_d7, 0) AS retained_d7

FROM eligible_users e
LEFT JOIN d7_target t USING (user_id);

## 5. Base Mart 필수 QA

### 5-1. 사용자 1명당 1행

In [ ]:
%%bigquery df_qa_dedup --project your-gcp-project

SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT user_id) AS user_count,
  COUNT(*) - COUNT(DISTINCT user_id) AS duplicate_count
FROM `your-gcp-project.mart.mart_ml_d7_base`;

In [ ]:
df_qa_dedup
# 정상: row_count = user_count, duplicate_count = 0

### 5-2. 시간창 중첩 · 관찰 부족 확인

In [ ]:
%%bigquery df_qa_window --project your-gcp-project

SELECT
  COUNTIF(feature_start_at >= feature_end_at) AS invalid_feature_window,
  COUNTIF(feature_end_at > target_start_at) AS overlapping_window,
  COUNTIF(target_start_at >= target_end_at) AS invalid_target_window,
  COUNTIF(target_end_at > observation_end_exclusive) AS incomplete_target_window,
  COUNTIF(retained_d7 IS NULL) AS null_target
FROM `your-gcp-project.mart.mart_ml_d7_base`;

In [ ]:
df_qa_window
# 모두 0이어야 정상

### 5-3. 타깃 분포

In [ ]:
%%bigquery df_qa_target_dist --project your-gcp-project

SELECT
  retained_d7,
  COUNT(*) AS user_count,
  ROUND(SAFE_DIVIDE(COUNT(*), SUM(COUNT(*)) OVER ()) * 100, 2) AS user_pct
FROM `your-gcp-project.mart.mart_ml_d7_base`
GROUP BY retained_d7
ORDER BY retained_d7;

In [ ]:
df_qa_target_dist

### 5-4. 최초 PING 기간 확인 (최종)

In [ ]:
%%bigquery df_qa_period --project your-gcp-project

SELECT
  MIN(first_ping_at_kst) AS min_first_ping_at_kst,
  MAX(first_ping_at_kst) AS max_first_ping_at_kst,
  COUNT(*) AS eligible_user_count
FROM `your-gcp-project.mart.mart_ml_d7_base`;

In [ ]:
df_qa_period

## 왜 기존 4,849명보다 N이 줄어들 수 있나

기존 쿼리의 4,849명은 원천 테이블 전체 기간에 PING을 한 번이라도 보낸 사용자였습니다. 이번 Base에서는
다음 두 조건이 추가돼서, N명이 4,849명보다 **작게 나오는 게 정상**입니다.

1. 2023-07-31 이후 최초 활동 사용자 제외
2. 최초 PING 이후 D+8까지 관찰할 수 없는 사용자 제외


## 6. Base가 만들어진 다음 — 역할 분담

Base Mart가 팀 검토를 통과해서 **동결**되면, 각 팀원은 반드시 이 테이블에서 시작합니다.

```sql
FROM `your-gcp-project.mart.mart_ml_d7_base` b
LEFT JOIN `your-gcp-project.raw.raw_userquestionrecord` q
  ON b.user_id = CAST(q.user_id AS INT64)
  AND q.created_at >= b.feature_start_at
  AND q.created_at < b.feature_end_at
```

힌트·질문 카테고리·포인트도 동일하게 **Base의 `user_id` + `feature_start_at` + `feature_end_at`**을
기준으로 계산합니다.

**역할 분담**
- 팀원 A: PING·관계망 피처
- 팀원 B: 힌트·Basic 피처
- 팀원 C: 질문 카테고리 피처
- 나머지: 포인트·질문세트 또는 최종 통합

## 다음 단계 (Base Mart 확정 후에만 진행)

- 각 피처 테이블 생성 → Base와 `user_id` 기준 조인해서 최종 학습 테이블 완성
- Train/Test 분할 (시간 기준 분할 원칙 유지)
- 스케일링, 모델 학습 (XGBoost)
- Optuna 하이퍼파라미터 튜닝
- SHAP으로 feature importance 해석
- 위험군 리스트 생성

**지금 순서는 정말 이것뿐입니다**: ① 관찰 종료일·시간대 확정 → ② 최종 후보 N명 계산 → ③ Base Mart 생성 →
④ 중복·관찰기간·타깃 분포 QA → ⑤ 팀에 Base Mart 공유

In [ ]:
%%bigquery --project your-gcp-project

CREATE OR REPLACE TABLE `your-gcp-project.mart.feat_user_basic_d7` AS

SELECT
  b.user_id,

  u.gender,

  g.school_id,
  g.group_id,
  g.grade,
  g.class_num,

  s.school_type,
  s.student_count

FROM `your-gcp-project.mart.mart_ml_d7_base` b
LEFT JOIN `your-gcp-project.stage.stg_users` u
  ON b.user_id = u.user_id
LEFT JOIN `your-gcp-project.stage.stg_group` g
  ON u.group_id = g.group_id
LEFT JOIN `your-gcp-project.stage.stg_school` s
  ON g.school_id = s.school_id;

In [ ]:
%%bigquery df_basic_qa --project your-gcp-project

SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT user_id) AS user_count,
  COUNTIF(gender IS NULL) AS null_gender,
  COUNTIF(school_type IS NULL) AS null_school_type,
  COUNTIF(grade IS NULL) AS null_grade
FROM `your-gcp-project.mart.feat_user_basic_d7`

In [ ]:
df_basic_qa
# row_count = user_count = 4,844 이어야 정상 (Base Mart와 행 수 일치)
# null_* 값들은 stg_users에 group_id가 없는 유저가 있으면 발생 가능 — 있다면 원인 확인 필요